<a href="https://colab.research.google.com/github/YoussefAli07/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [20]:
# !git clone https://github.com/YoussefAli07/flyrank-ml-internship-starter.git

In [21]:
import pandas as pd
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: "High search volume is a weak expectation-setting number"
**Claim:** The paper reports a near-zero correlation between search volume and impressions, concluding that search volume weakly predicts traffic.

**Methodology question:** This correlation is computed only on the "local active-content sample," which is filtered to `impressions_90d > 0 AND sessions_90d > 0`. That filter excludes, by construction, any high-search-volume page that earned *zero* traffic, which is exactly the failure mode a reader would most want this finding to speak to. The claim as tested is closer to "among pages that already got some traffic, search volume doesn't predict how much more" — not "search volume doesn't predict whether a page gets traffic at all." Was the zero-traffic population checked separately before generalizing the claim to the whole portfolio?

### Finding 2: "Click capture by position tier"
**Claim:** Weighted CTR falls sharply as position tier moves away from the top of search results — roughly an 88% drop from Top 3 (0.423%) to Deep (0.050%).

**Methodology question:** The paper doesn't disclose whether rows with `avg_position = 0` - which represent *missing* position data, not an actual top rank - were excluded before pages were bucketed into tiers. If any such rows landed in a tier (most plausibly "Deep," since 0
would sort as the lowest numeric value), their clicks/impressions would distort that tier's weighted CTR without reflecting real ranking behavior at all. Confirming this exclusion happened would strengthen an otherwise well-aggregated (weighted-CTR-over-per-row-average) methodology.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Setup:** Same features, target (`trend_pct`), model config (Decision Tree, max_depth=4),
and target-capping rule (train-only, 1st/99th percentile) as ML-08. Only the split method
changes between the two runs below, everything else is held identical so any difference
in the numbers is attributable to the split, not to a different pipeline.

| Split type | Naive MAE | Model MAE | Beats naive by | % improvement |
|---|---|---|---|---|
| Honest (grouped by client_id) | 68.15 | 65.24 | 2.91 | 4.3% |
| Dishonest (random, no grouping) | 69.44 | 63.46 | 5.99 | 8.6% |

**Interpretation:** The dishonest split makes the model look almost twice as good as it
really is (8.6% vs. 4.3% improvement over the naive baseline). This gap is the leakage
mechanism we reasoned through earlier: a random split lets rows from the same client land
in both train and test. Since the model never sees `client_id` directly, it isn't
memorizing clients by name, it's matching feature-space patterns (word_count,
search_volume, competition, content_type, main_intent combinations) that happen to be
distinctive to a given client. When some of that client's rows are in train and others in
test, the model is partly grading itself on data it has effectively already seen the answer
for, rather than generalizing to a genuinely new client.

The honest number (4.3%) is the one that should inform real decisions. It reflects
performance on clients the model has never encountered, which is the actual deployment
scenario for this model. The dishonest number (8.6%) would overstate the model's real-world
value if reported on its own.

*Note: an earlier version of this comparison used the training-mean after outlier
capping as the naive baseline, which understated the baseline and inflated the apparent
"beats naive" margin for both splits. The naive baseline here uses the raw, uncapped
training mean. The model itself still trains on capped data, a separate and legitimate
choice.*

In [22]:
feature_cols = ['search_volume', 'competition', 'content_type', 'main_intent', 'word_count']
X = df[feature_cols]
y = df['trend_pct']
X = pd.get_dummies(X, columns=['content_type', 'main_intent'])

In [23]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

# HONEST split: grouped by client_id, same as ML-08
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=df['client_id']))
X_train_g, X_test_g = X.iloc[train_idx], X.iloc[test_idx]
y_train_g, y_test_g = y.iloc[train_idx], y.iloc[test_idx]

# Drop missing target rows
valid_train_g = y_train_g.notnull()
X_train_g, y_train_g = X_train_g[valid_train_g], y_train_g[valid_train_g]
valid_test_g = y_test_g.notnull()
X_test_g, y_test_g = X_test_g[valid_test_g], y_test_g[valid_test_g]


lower_cap_g = y_train_g.quantile(0.01)
upper_cap_g = y_train_g.quantile(0.99)
y_train_g_capped = y_train_g.clip(lower=lower_cap_g, upper=upper_cap_g)


naive_pred_g = y_train_g.mean()
naive_mae_g = mean_absolute_error(y_test_g, np.full(len(y_test_g), naive_pred_g))

# Model trains on the capped target (legitimate, disclosed modeling choice)
tree_model_g = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_model_g.fit(X_train_g, y_train_g_capped)
tree_mae_g = mean_absolute_error(y_test_g, tree_model_g.predict(X_test_g))

print("=== HONEST split (grouped by client_id) ===")
print("Naive baseline MAE (uncapped mean):", naive_mae_g)
print("Decision Tree MAE (capped training):", tree_mae_g)
print("Beats naive by:                     ", naive_mae_g - tree_mae_g)

=== HONEST split (grouped by client_id) ===
Naive baseline MAE (uncapped mean): 68.14762960803775
Decision Tree MAE (capped training): 65.24051816119345
Beats naive by:                      2.907111446844297


In [24]:
from sklearn.model_selection import train_test_split

# DISHONEST split: random, no client grouping — rows from the same
# client can end up in both train and test.
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)


valid_train_r = y_train_r.notnull()
X_train_r, y_train_r = X_train_r[valid_train_r], y_train_r[valid_train_r]
valid_test_r = y_test_r.notnull()
X_test_r, y_test_r = X_test_r[valid_test_r], y_test_r[valid_test_r]


lower_cap_r = y_train_r.quantile(0.01)
upper_cap_r = y_train_r.quantile(0.99)
y_train_r_capped = y_train_r.clip(lower=lower_cap_r, upper=upper_cap_r)


naive_pred_r = y_train_r.mean()
naive_mae_r = mean_absolute_error(y_test_r, np.full(len(y_test_r), naive_pred_r))


tree_model_r = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_model_r.fit(X_train_r, y_train_r_capped)
tree_mae_r = mean_absolute_error(y_test_r, tree_model_r.predict(X_test_r))

print("=== DISHONEST split (random, no client grouping) ===")
print("Naive baseline MAE (uncapped mean):", naive_mae_r)
print("Decision Tree MAE (capped training):", tree_mae_r)
print("Beats naive by:                     ", naive_mae_r - tree_mae_r)

=== DISHONEST split (random, no client grouping) ===
Naive baseline MAE (uncapped mean): 69.44322307782362
Decision Tree MAE (capped training): 63.45631108412796
Beats naive by:                      5.9869119936956565


In [25]:
pct_g = (naive_mae_g - tree_mae_g) / naive_mae_g * 100
pct_r = (naive_mae_r - tree_mae_r) / naive_mae_r * 100

print(f"{'Split type':<25}{'Naive MAE':>12}{'Model MAE':>12}{'Beats naive by':>18}{'% improvement':>16}")
print(f"{'Honest (grouped)':<25}{naive_mae_g:>12.2f}{tree_mae_g:>12.2f}{naive_mae_g - tree_mae_g:>18.2f}{pct_g:>15.1f}%")
print(f"{'Dishonest (random)':<25}{naive_mae_r:>12.2f}{tree_mae_r:>12.2f}{naive_mae_r - tree_mae_r:>18.2f}{pct_r:>15.1f}%")

Split type                  Naive MAE   Model MAE    Beats naive by   % improvement
Honest (grouped)                68.15       65.24              2.91            4.3%
Dishonest (random)              69.44       63.46              5.99            8.6%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Ran the ML-05/hunting-leakage checklist against the final feature set
(`search_volume`, `competition`, `content_type`, `main_intent`, `word_count` → `trend_pct`):

- **Label-derived features:** none. `ctr`, `avg_position`, `trend_direction` were already
  excluded in ML-03/04 as label-derived from `trend_pct`.
- **Future/overlapping windows:** none. all 5 features are static content attributes, not
  time-windowed aggregates that could overlap the label's measurement window.
- **Product flags:** none of the 5 features are FlyRank-generated decision flags.
- **Grouped split:** Done in Section 2 (client_id, zero overlap).
- **Base rate / naive baseline:** printed alongside every metric (Section 2).

**Feature importance investigated (the one real flag from ML-08):** `word_count` showed
98.4% importance in ML-08, flagged there as suspicious but not tested. Ran
with/without test:

| Configuration | MAE |
|---|---|
| Naive baseline | 68.15 |
| WITH word_count | 65.24 |
| WITHOUT word_count | 64.90 |

**Verdict: not leakage.** A genuinely leaking feature collapses model performance when
removed (the skill's benchmark: ~1.0 → ~0.7). Here, removing `word_count` barely changed
MAE, if anything it's better without it. This confirms the alternate hypothesis
already raised in ML-08: the shallow tree (`max_depth=4`) has only a few splits to work
with, and grabbing `word_count` as an early split concentrates its importance score
without that feature carrying anywhere near 98.4% of the real predictive signal. It's a
matter of tree shallowness, not a leaking or dominant feature.

In [26]:
# Leakage check: does the model collapse without word_count?

feature_cols_no_wc = ['search_volume', 'competition', 'content_type', 'main_intent']
X_no_wc = pd.get_dummies(df[feature_cols_no_wc], columns=['content_type', 'main_intent'])


X_train_nowc = X_no_wc.loc[X_train_g.index]
X_test_nowc = X_no_wc.loc[X_test_g.index]

tree_nowc = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_nowc.fit(X_train_nowc, y_train_g_capped)
mae_nowc = mean_absolute_error(y_test_g, tree_nowc.predict(X_test_nowc))

print("MAE WITH word_count:   ", tree_mae_g)
print("MAE WITHOUT word_count:", mae_nowc)
print("Naive baseline:        ", naive_mae_g)

MAE WITH word_count:    65.24051816119345
MAE WITHOUT word_count: 64.89669505386065
Naive baseline:         68.14762960803775


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (overclaimed):** "word_count is the dominant driver of trend_pct performance."

**Problem:** This claim would be a natural takeaway from ML-08's raw feature importance
output (98.4%), but Section 3's leakage audit showed that removing word_count barely
changes model MAE (65.24 → 64.90). The importance score reflects a shallow-tree structural
artifact, not a true dominant relationship, so the original sentence overstates what was
actually measured.

**Rewritten (safe language):** "In this shallow Decision Tree (max_depth=4), word_count was
observed to receive the largest share of feature importance (98.4%), but a controlled
with/without comparison shows this is directional evidence of tree structure, not of
word_count's real predictive weight and removing it changed MAE by less than 1%. Any decision
to prioritize word_count as a content lever should rely on the direct portfolio-level
findings, not on this model's importance scores alone."

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.